## hex/decimal data translate

In [2]:
def hex_to_signed_int(hex_str, bits):
    value = int(hex_str, 16)
    if value >= (1 << (bits - 1)):
        value -= (1 << bits)
    return value

def hex_txt_to_dec(file_path, bits, start_line=0, end_line=None, hex_chars_per_item=4):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    selected_lines = lines[start_line:end_line] if end_line else lines[start_line:]

    tensor_decimal = []
    for line in selected_lines:
        line = line.strip()
        for i in range(0, len(line), hex_chars_per_item):
            hex_str = line[i:i + hex_chars_per_item]
            if hex_str:  # 避免空字串
                tensor_decimal.append(hex_to_signed_int(hex_str, bits=bits))

    return tensor_decimal


In [3]:
import numpy as np
def hex_txt_to_dec_cnn(
    file_path,
    bits=16,
    ker_L=3,
    out_ch=1,
    start_line=0,
    end_line=None
):
    """
    從 file_path 讀 hex 權重檔（每行一串 hex),
    bits            : 每筆權重的位元長度 (如 16)
    ker_L           : 1D kernel 長度
    out_ch          : output channels
    start_line,end_line : 要讀的行範圍（預設讀完所有行）
    hex_chars_per_item  : 每筆資料佔多少個 hex 字元 (預設 4)
    回傳 shape = (out_ch, in_ch, ker_L) 的 numpy array
    """
    # 1. 讀檔並轉成有號整數列表
    hex_chars_per_item = int(bits/4)
    with open(file_path, 'r') as f:
        lines = f.readlines()
    lines = lines[start_line:end_line] if end_line else lines[start_line:]
    data = []
    for line in lines:
        s = line.strip()
        for i in range(0, len(s), hex_chars_per_item):
            h = s[i:i+hex_chars_per_item]
            if h:
                data.append(hex_to_signed_int(h, bits))

    # 2. 計算 in_ch、reshape
    arr = np.array(data, dtype=np.int32)
    total = arr.size
    per_out = ker_L * out_ch
    if total % per_out != 0:
        raise ValueError(f"總數 {total} 無法整除 ker_L={ker_L} * out_ch={out_ch}")
    in_ch = total // per_out
    return arr.reshape((out_ch, in_ch, ker_L))

Print ifmap, kernel, ofmap

In [4]:
conv1_ifm = hex_txt_to_dec_cnn(
    file_path='../Data/ecg_v2/input/input1.txt',
    bits=16,
    out_ch=1,
    ker_L=256)
print(conv1_ifm)

[[[ -81  -79  -79  -80  -82  -83  -83  -81  -79  -79  -80  -81  -83  -83
    -82  -81  -81  -81  -80  -78  -76  -72  -69  -66  -61  -54  -47  -40
    -36  -33  -32  -28  -22  -14   -7   -4   -5   -7   -8   -7   -5   -5
    -11  -23  -37  -47  -50  -45  -37  -31  -29  -34  -43  -52  -62  -70
    -79  -88  -96 -102 -106 -109 -112 -116 -120 -123 -125 -125 -126 -128
   -131 -133 -132 -129 -125 -124 -125 -127 -129 -129 -126 -123 -124 -132
   -146 -167 -193 -221 -246 -258 -245 -192  -90   63  265  506  773 1043
   1287 1465 1535 1464 1249  921  543  191  -70 -210 -239 -196 -133  -84
    -65  -70  -84  -95  -99 -101 -106 -116 -128 -139 -147 -151 -154 -157
   -161 -164 -166 -166 -166 -167 -169 -171 -171 -171 -169 -169 -169 -170
   -171 -171 -169 -166 -164 -161 -159 -157 -155 -154 -153 -152 -149 -145
   -139 -133 -129 -127 -126 -123 -119 -113 -106  -99  -92  -86  -79  -70
    -61  -50  -40  -30  -20   -8    5   19   33   46   60   73   87  100
    112  123  134  145  158  171  183  193  202  21

In [5]:
conv1_ker = hex_txt_to_dec_cnn(
    file_path='../Data/ecg_v2/weight/weight.txt',
    bits=16,
    start_line=0,
    end_line=16,
    ker_L=3,
    out_ch=16
    )
print(conv1_ker)

[[[  -53   -10  -163]]

 [[  155   -28    98]]

 [[ -110   -72   436]]

 [[  -61   -19  -175]]

 [[   25  -109  -163]]

 [[   25   -74  -182]]

 [[   46   113    57]]

 [[  462 -1211   528]]

 [[   -4  -121  -144]]

 [[  271  -156    99]]

 [[  103    76    76]]

 [[  -88   -39   369]]

 [[  307    77  -131]]

 [[ -673    27   440]]

 [[ -134   -52   -61]]

 [[  367  -202  -407]]]


In [4]:
def conv1d(ifmap, weight, bias=None, stride=1, padding=0):
    """
    ifmap  : numpy array, shape = (batch, in_ch, L_in)
    weight : numpy array, shape = (out_ch, in_ch, ker_L)
    bias   : numpy array or None, shape = (out_ch,)
    stride : int, 步距
    padding: int or tuple, 左右兩端補零 (int 表示左右相同，tuple 表示 (left, right))
    return : numpy array, shape = (batch, out_ch, L_out)
    """
    batch, in_ch, L_in = ifmap.shape
    out_ch, _, ker_L = weight.shape

    # 處理 padding 為 int 或 tuple
    if isinstance(padding, int):
        pad_left = pad_right = padding
    else:
        pad_left, pad_right = padding

    # 在時間軸左右補零
    x = np.pad(ifmap,
               pad_width=((0,0), (0,0), (pad_left, pad_right)),
               mode='constant',
               constant_values=0)

    L_out = (L_in + pad_left + pad_right - ker_L) // stride + 1
    out = np.zeros((batch, out_ch, L_out), dtype=ifmap.dtype)

    for b in range(batch):
        for o in range(out_ch):
            for i in range(L_out):
                start = i * stride
                end   = start + ker_L
                window = x[b, :, start:end]
                out[b, o, i] = np.sum(window * weight[o]) + (bias[o] if bias is not None else 0)
    return out


In [6]:
conv1_out = conv1d(ifmap=conv1_ifm, weight=conv1_ker, stride=1)
print(conv1_out)

[[[ 17960  18017  18353 ...   6623   5998   5483]
  [-18085 -17873 -18041 ...  -6991  -6471  -5894]
  [-19846 -20502 -21302 ...  -6346  -5474  -5148]
  ...
  [ 17620  15834  14927 ...   9052   9107   8049]
  [ 19781  19574  19748 ...   7742   7135   6455]
  [ 18384  19525  20541 ...   5547   4454   4135]]]


## DLA Instruction Set Compiler

In [5]:
def dla_inst_gen(
    conv=1,
    ifm_ch=0,
    ifm_id=0,
    ker_w=2,
    ker_addr=0,
    ofm_ch=15,
    ofm_L=127,
    mp_l=0,
    gap=0,
    padding=1,
    stride=1,
    ReLU=1,
    bias_true=1,
    base=16
):
    '''
    name        max_value       bits
    conv                        4
    ifm_ch      16              5
    ifm_id                      1
    ker_w       7               3
    ker_addr    1023            10
    ofm_ch      16              5
    ofm_L       128             9
    mp_l        3               2
    gap                         1
    padding     1,1             2
    stride      7               2
    ReLU                        1
    bias_true                   1
    total                       
    '''
    # 每個欄位的順序與其對應的 bit 數
    fields = [
        ("conv", conv, 4),
        ("ifm_ch", ifm_ch, 4),
        ("ifm_id", ifm_id, 1),
        ("ker_w", ker_w, 2),
        ("ker_addr", ker_addr, 10),
        ("ofm_ch", ofm_ch, 4),
        ("ofm_L", ofm_L, 9),
        ("mp_l", mp_l, 2),
        ("gap", gap, 1),
        ("padding", padding, 2),
        ("stride", stride, 2),
        ("ReLU", ReLU, 1),
        ("bias_true", bias_true, 1)
    ]
    inst = 0
    total_bits = 0
    for name, value, bits in fields:
        inst = (inst << bits) | (value & ((1 << bits) - 1))
        total_bits += bits
    # 若未滿 64 bits，補上 0
    if total_bits <= 64:
        inst = inst << (64 - total_bits)
    else:
        raise ValueError(f"Total bits exceed 64! The current field {name} would cause overflow.")
    match base:
        case 2:
            return format(inst, '064b')
        case 16:
            return format(inst, '016X')
        case _:
            return inst


In [8]:
inst_conv0 = dla_inst_gen(conv=0, base=16)
print(inst_conv0)

0040079FC2E00000


## conv1d simulator

In [21]:
import numpy as np

def conv1d_debug(inputs, weights, bias=None, stride=1):
    """
    1D convolution with debug print for each MAC step.
    
    inputs: numpy array with shape (C_in, L_in)
    weights: numpy array with shape (C_out, C_in, K)
    bias: int, hex string, or numpy array (C_out,) or None
    stride: int, default 1
    """
    C_in, L_in = inputs.shape
    C_out, C_in_w, K = weights.shape
    assert C_in == C_in_w, "Input channel 與 weight channel 數量不一致"

    L_out = (L_in - K) // stride + 1
    outputs = np.zeros((C_out, L_out), dtype=np.int32)

    # 處理 bias，自動轉成 signed 32-bit
    if bias is None:
        bias = np.zeros(C_out, dtype=np.int32)
    else:
        if isinstance(bias, (int, str)):
            b = int(bias, 16) if isinstance(bias, str) else int(bias)
            b = b - 0x100000000 if b >= 0x80000000 else b
            bias = np.array([b], dtype=np.int32)
        else:
            new_bias = []
            for b in bias:
                b = int(b, 16) if isinstance(b, str) else int(b)
                b = b - 0x100000000 if b >= 0x80000000 else b
                new_bias.append(b)
            bias = np.array(new_bias, dtype=np.int32)

    # convolution 運算
    for co in range(C_out):
        for lo in range(L_out):
            res = 0
            print(f"\n[Output channel {co}, position {lo}, stride={stride}]")
            for ci in range(C_in):
                for k in range(K):
                    inp = int(inputs[ci, lo*stride + k])
                    wgt = int(weights[co, ci, k])
                    # sign extend (16-bit)
                    inp = inp - 0x10000 if inp >= 0x8000 else inp
                    wgt = wgt - 0x10000 if wgt >= 0x8000 else wgt
                    mul = inp * wgt
                    res += mul
                    print(f"  ci={ci}, k={k}, inp={hex(inp & 0xFFFF)}, "
                          f"wgt={hex(wgt & 0xFFFF)}, mul={hex(mul & 0xFFFFFFFF)}, "
                          f"psum={hex(res & 0xFFFFFFFF)}")
            res += int(bias[co])
            outputs[co, lo] = res & 0xFFFFFFFF
            print(f"  + bias={hex(int(bias[co]) & 0xFFFFFFFF)} => "
                  f"ofm={hex(outputs[co, lo])}")

    return outputs


In [24]:
# ===== 測試範例 =====
inputs_0 = np.array([[0xFFAF, 0xFFB1, 0xFFB1, 0xFFB0, 0xFFAE]], dtype=np.uint16)

weights_0_0 = np.array([[[0xFFCB, 0xFFF6, 0xFF5D]]], dtype=np.uint16)
weights_0_4 = np.array([[[0x0019, 0xFF93, 0xFF5D]]], dtype=np.uint16)

bias_0 = "0xFFFFF400"
bias_4 = "0x00004200"

# 控制 stride，例如 stride=2
outputs_0_0 = conv1d_debug(inputs_0, weights_0_0, bias=bias_0, stride=2)
outputs_0_4 = conv1d_debug(inputs_0, weights_0_4, bias=bias_4, stride=2)
print("\n=== Final OFM ===")


[Output channel 0, position 0, stride=2]
  ci=0, k=0, inp=0xffaf, wgt=0xffcb, mul=0x10c5, psum=0x10c5
  ci=0, k=1, inp=0xffb1, wgt=0xfff6, mul=0x316, psum=0x13db
  ci=0, k=2, inp=0xffb1, wgt=0xff5d, mul=0x324d, psum=0x4628
  + bias=0xfffff400 => ofm=0x3a28

[Output channel 0, position 1, stride=2]
  ci=0, k=0, inp=0xffb1, wgt=0xffcb, mul=0x105b, psum=0x105b
  ci=0, k=1, inp=0xffb0, wgt=0xfff6, mul=0x320, psum=0x137b
  ci=0, k=2, inp=0xffae, wgt=0xff5d, mul=0x3436, psum=0x47b1
  + bias=0xfffff400 => ofm=0x3bb1

[Output channel 0, position 0, stride=2]
  ci=0, k=0, inp=0xffaf, wgt=0x19, mul=0xfffff817, psum=0xfffff817
  ci=0, k=1, inp=0xffb1, wgt=0xff93, mul=0x21a3, psum=0x19ba
  ci=0, k=2, inp=0xffb1, wgt=0xff5d, mul=0x324d, psum=0x4c07
  + bias=0x4200 => ofm=0x8e07

[Output channel 0, position 1, stride=2]
  ci=0, k=0, inp=0xffb1, wgt=0x19, mul=0xfffff849, psum=0xfffff849
  ci=0, k=1, inp=0xffb0, wgt=0xff93, mul=0x2210, psum=0x1a59
  ci=0, k=2, inp=0xffae, wgt=0xff5d, mul=0x3436, psum